# Stage 5 — CPU Streamlit application verification

Branch: **feat/streamlit-deployment**. Use a fresh Colab notebook session with
**CPU runtime** (no GPU needed). Run cells individually. The notebook creates
an isolated Python 3.13 environment with the same pinned CPU dependencies intended
for Streamlit Cloud; it does not change the earlier Stage 4 environment.

Tests run first using fake models. Real predictions then check Small → Base →
Large → Small on one image. These checks do not prove Community Cloud has enough
memory for every model. Public configuration stays Small-only until hosted review.
No training, dataset download, report editing, or benchmark rerun is required.

## 1. Create a clean checkout and isolated CPU environment

In [ ]:
from pathlib import Path
from google.colab import files
import ast, datetime, json, os, platform, subprocess, sys, tempfile, zipfile

assert sys.version_info[:2] == (3, 13), "Use Python 3.13 to match the planned hosting configuration; return this output if unavailable."
BRANCH = "feat/streamlit-deployment"
workspace = Path(tempfile.mkdtemp(prefix="dav2_stage5_", dir="/content"))
ROOT = workspace / "repo"
EVIDENCE = workspace / "evidence"
EVIDENCE.mkdir()

def command_logged(command, log_path):
    env = dict(os.environ)
    env["PYTHONPATH"] = str(ROOT) + os.pathsep + str(ROOT / "src")
    env["PYTHONUNBUFFERED"] = "1"
    # Colab inline plotting is unavailable in this isolated headless environment.
    env["MPLBACKEND"] = "Agg"
    with Path(log_path).open("w", encoding="utf-8") as log:
        process = subprocess.Popen(command, cwd=ROOT if ROOT.exists() else workspace,
                                   env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                   text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
                log.write(line)
                log.flush()
            code = process.wait()
        except BaseException:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if code:
        raise RuntimeError(f"Command failed with exit code {code}. Return {log_path.name}; do not continue.")

command_logged(["git", "clone", "--branch", BRANCH, "--single-branch",
                "https://github.com/ayyansohail/depth-anything-v2-reproduction.git", str(ROOT)],
               EVIDENCE / "checkout.log")
VENV = workspace / "cpu_environment"
environment_builder = "stdlib venv"
try:
    command_logged([sys.executable, "-m", "venv", str(VENV)], EVIDENCE / "venv.log")
except RuntimeError:
    print("Standard venv/pip bootstrap failed; using a separate virtualenv bootstrap. The failed environment and log are preserved.")
    BOOTSTRAP = workspace / "virtualenv_bootstrap"
    command_logged([sys.executable, "-m", "pip", "install", "--target", str(BOOTSTRAP), "virtualenv==20.35.4"],
                   EVIDENCE / "virtualenv_installation.log")
    VENV = workspace / "cpu_environment_fallback"
    command_logged([sys.executable, "-c",
                    "import sys; sys.path.insert(0, sys.argv[1]); from virtualenv import cli_run; cli_run([sys.argv[2], '--no-download'])",
                    str(BOOTSTRAP), str(VENV)], EVIDENCE / "virtualenv.log")
    environment_builder = "virtualenv 20.35.4 fallback"
PYTHON = str(VENV / "bin/python")
command_logged([PYTHON, "-m", "pip", "--version"], EVIDENCE / "pip_bootstrap.log")
command_logged([PYTHON, "-m", "pip", "install", "-r", str(ROOT / "app/requirements.txt"), "pytest==8.4.2"],
               EVIDENCE / "installation.log")
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
(EVIDENCE / "checkout.json").write_text(json.dumps({"branch": BRANCH, "commit": commit,
    "python": platform.python_version(), "environment_builder": environment_builder, "context": "new Colab isolated CPU application verification"}, indent=2))
print("Prepared:", BRANCH, commit)

## 2. Parse source and run helper, model-loader and UI tests

Tests use fake predictions and do not download model weights. Stop on failure
and return its log. Existing Stage 3 metric tests are included as regression checks.

In [ ]:
assert "PYTHON" in globals(), "Run setup in this session first."
for folder in ("app", "tests", "src"):
    for path in (ROOT / folder).rglob("*.py"):
        ast.parse(path.read_text(encoding="utf-8-sig"), filename=str(path))
ast.parse((ROOT / "scripts/verify_deployment.py").read_text(encoding="utf-8-sig"))
notebook = json.loads((ROOT / "notebooks/stage5_deployment_verification.ipynb").read_text(encoding="utf-8-sig"))
for index, cell in enumerate(notebook["cells"]):
    if cell["cell_type"] == "code":
        ast.parse("".join(cell["source"]), filename=f"stage5_cell_{index}")
command_logged([PYTHON, "-m", "pytest", "-q", "tests/test_metrics.py", "tests/test_deployment.py"],
               EVIDENCE / "tests.log")
print("Parsing and tests passed. Proceed to upload one example photo.")

## 3. Upload one worked-example photo

Choose one JPG or PNG you are happy to include with the course evidence.
The evidence ZIP contains its resized input and generated maps. Requirements:
up to 10 MB, 16 megapixels, aspect ratio at most 2:1. File names may contain
numbered download suffixes; no special filename is required.

In [ ]:
uploaded = files.upload()
assert len(uploaded) == 1, "Select exactly one JPG/PNG photo, then rerun this upload cell."
name, content = next(iter(uploaded.items()))
safe_name = Path(name).name
INPUT = workspace / safe_name
INPUT.write_bytes(content)
print("Using:", safe_name)

## 4. Verify real CPU predictions and model switching

This downloads pinned model weights into the Colab cache. It runs Small, Base,
Large, then Small again. Larger CPU models can take time. Output validation,
retained-result checks, loading metadata and a dependency snapshot are saved.
This does not enable Base or Large on the public deployment.

In [ ]:
assert "INPUT" in globals(), "Upload one example image first."
command_logged([PYTHON, "scripts/verify_deployment.py", "--image", str(INPUT),
                "--out-dir", str(EVIDENCE / "real_predictions")],
               EVIDENCE / "real_predictions_console.log")
from IPython.display import Image, display
for path in sorted((EVIDENCE / "real_predictions").glob("*_colour_png.png")):
    display(Image(filename=str(path)))

## 5. Download verification evidence and stop for review

Run this even if a previous step failed (provided setup created workspace and
EVIDENCE). The ZIP excludes the virtual environment, model cache and original
full-resolution upload. Return it in the chat before deploying.

In [ ]:
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
archive = workspace / f"stage5_verification_{stamp}.zip"
with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
    for path in sorted(EVIDENCE.rglob("*")):
        if path.is_file():
            z.write(path, str(path.relative_to(EVIDENCE)))
print("Evidence:", archive)
files.download(str(archive))
print("STOP: return this ZIP for review. Hosting instructions follow only after verification.")

## After review: Streamlit Cloud

The app will initially expose Small only. Select repository
ayyansohail/depth-anything-v2-reproduction, branch feat/streamlit-deployment,
entrypoint app/streamlit_app.py, Python 3.13. See docs/DEPLOYMENT.md for the exact
workflow. Colab passes do not count as hosted acceptance. The owner returns the
live URL, any errors, and a screenshot after a real hosted prediction.